Una pequeña sucursal de un banco tiene dos empleados, uno para los pagos y otro para los cobros. Los clientes llegan a cada caja siguiendo una distribución de Poisson con una media de 20/hora. (el total de llegada al banco es de 40/hora). El tiempo de servicio de cada empleado es una negativa exponencial de media 2 minutos. El encargado de la sección está pensando hacer un cambio en que los dos operarios puedan hacer tanto pagos como cobros para evitar situaciones en que una cola está llena y la otra parada. Sin embargo, se estima que cuando los empleados se encarguen de las dos cosas el tiempo de servicio aumentará a una media de 2,4 minutos. Compara el sistema que se emplea ahora con el propuesto, calculando el total de gente en el banco, el tiempo medio que pasaría un cliente en el banco hasta que es atendido, la probabilidad de que un cliente espere más de cinco minutos y el tiempo medio que están parados los empleados.

In [ ]:
import simpy
import random
import statistics
import matplotlib.pyplot as plt
import numpy as np

# Parámetros generales
SIM_TIME = 10 * 60  # duración de la simulación en minutos (10 horas = 600 minutos)
SEED = 42
random.seed(SEED)

# ===============================
# Sistema 1: Dos colas independientes (M/M/1)
# Cada uno recibe llegadas Poisson con media 20/hora (1 llegada cada 3 minutos en promedio)
# y atiende con tiempo exponencial de media 2 minutos (μ = 30/hora)

ARRIVAL_RATE_S1 = 20/60   # llegadas por minuto (20/hora)
SERVICE_MEAN_S1   = 2       # minutos

class ClienteS1:
    def __init__(self, name, llegada):
        self.name = name
        self.llegada = llegada
        self.espera = None  # tiempo que espera en cola
        self.servicio_inicio = None
        self.salida = None  # tiempo de salida (fin de servicio)

def proceso_cliente_s1(env, cliente, servidor, tiempos_espera):
    """Proceso de un cliente en el sistema 1.
    Se registra el tiempo de inicio de servicio y se espera el tiempo de servicio."""
    with servidor.request() as req:
        yield req
        cliente.servicio_inicio = env.now
        espera = cliente.servicio_inicio - cliente.llegada
        cliente.espera = espera
        tiempos_espera.append(espera)
        # Tiempo de servicio (exponencial con media SERVICE_MEAN_S1)
        servicio = random.expovariate(1.0/SERVICE_MEAN_S1)
        yield env.timeout(servicio)
        cliente.salida = env.now

def generador_clientes_s1(env, servidor, tiempos_espera, clientes):
    """Generador de clientes para el sistema 1."""
    i = 0
    while env.now < SIM_TIME:
        inter_arrival = random.expovariate(ARRIVAL_RATE_S1)
        yield env.timeout(inter_arrival)
        i += 1
        cliente = ClienteS1(f"Cliente_{i}", env.now)
        clientes.append(cliente)
        env.process(proceso_cliente_s1(env, cliente, servidor, tiempos_espera))

def simular_sistema1():
    # Se crean dos entornos independientes, uno para cada servicio.
    resultados = {}
    datos = {}
    for caja in [1,2]:
        env = simpy.Environment()
        servidor = simpy.Resource(env, capacity=1)  # cada caja es M/M/1
        tiempos_espera = []  # para medir tiempo en cola
        clientes = []
        env.process(generador_clientes_s1(env, servidor, tiempos_espera, clientes))
        env.run(until=SIM_TIME)
        # Para la media en el sistema, se puede calcular el promedio del número de clientes simultáneos.
        # En este ejemplo, almacenaremos el tiempo de espera y el tiempo total (servicio+espera)
        tiempos_totales = [c.salida - c.llegada for c in clientes if c.salida is not None]
        # Para inactividad, la disponibilidad del servidor:
        # En un sistema M/M/1, el tiempo ocupado se suma a los tiempos de servicio.
        total_servicio = sum([c.salida - c.servicio_inicio for c in clientes if c.salida is not None])
        idle_time = SIM_TIME - total_servicio
        datos[f"Caja_{caja}"] = {
            "num_clientes": len(clientes),
            "promedio_espera": statistics.mean(tiempos_espera) if tiempos_espera else 0,
            "promedio_total": statistics.mean(tiempos_totales) if tiempos_totales else 0,
            "prob_espera_mas_5": sum(1 for w in tiempos_espera if w > 5)/len(tiempos_espera) if tiempos_espera else 0,
            "idle_time": idle_time
        }
    # Agregamos resultados totales (suma de ambas cajas)
    total_clientes = datos["Caja_1"]["num_clientes"] + datos["Caja_2"]["num_clientes"]
    promedio_total_tiempo = (datos["Caja_1"]["promedio_total"] * datos["Caja_1"]["num_clientes"] +
                             datos["Caja_2"]["promedio_total"] * datos["Caja_2"]["num_clientes"]) / total_clientes
    prob_espera_mas_5 = (datos["Caja_1"]["prob_espera_mas_5"] * datos["Caja_1"]["num_clientes"] +
                         datos["Caja_2"]["prob_espera_mas_5"] * datos["Caja_2"]["num_clientes"]) / total_clientes
    promedio_idle = (datos["Caja_1"]["idle_time"] + datos["Caja_2"]["idle_time"]) / 2.0
    # Aproximación del número medio de clientes en el sistema: usando Little, L = λ * W_total (en minutos)
    L_total = (ARRIVAL_RATE_S1 * total_clientes/ (20/60)) * promedio_total_tiempo  # simplificado
    # Aquí en vez de calcular L directamente de la simulación (guardando el número en cada instante), usamos las medias teóricas.
    resultados["Sistema1"] = {
        "total_clientes": total_clientes,
        "promedio_tiempo_total": promedio_total_tiempo,
        "prob_espera_mas_5": prob_espera_mas_5,
        "promedio_idle_empleado": promedio_idle
    }
    resultados["Detalle"] = datos
    return resultados

In [ ]:
# Sistema 2: Una única cola compartida por dos empleados (M/M/2)
# Llegadas totales: 40/hora, es decir 40/60 = 0.667 clientes por minuto.
# Tiempo de servicio: exponencial con media 2.4 minutos (μ = 25/hora).

ARRIVAL_RATE_S2 = 40/60   # llegadas por minuto
SERVICE_MEAN_S2   = 2.4     # minutos

class ClienteS2:
    def __init__(self, name, llegada):
        self.name = name
        self.llegada = llegada
        self.espera = None
        self.servicio_inicio = None
        self.salida = None

def proceso_cliente_s2(env, cliente, servidor, tiempos_espera):
    with servidor.request() as req:
        yield req
        cliente.servicio_inicio = env.now
        espera = cliente.servicio_inicio - cliente.llegada
        cliente.espera = espera
        tiempos_espera.append(espera)
        servicio = random.expovariate(1.0/SERVICE_MEAN_S2)
        yield env.timeout(servicio)
        cliente.salida = env.now

def generador_clientes_s2(env, servidor, tiempos_espera, clientes):
    i = 0
    while env.now < SIM_TIME:
        inter_arrival = random.expovariate(ARRIVAL_RATE_S2)
        yield env.timeout(inter_arrival)
        i += 1
        cliente = ClienteS2(f"Cliente_{i}", env.now)
        clientes.append(cliente)
        env.process(proceso_cliente_s2(env, cliente, servidor, tiempos_espera))

def simular_sistema2():
    env = simpy.Environment()
    # Recurso con 2 servidores
    servidor = simpy.Resource(env, capacity=2)
    tiempos_espera = []
    clientes = []
    env.process(generador_clientes_s2(env, servidor, tiempos_espera, clientes))
    env.run(until=SIM_TIME)
    
    tiempos_totales = [c.salida - c.llegada for c in clientes if c.salida is not None]
    total_servicio = sum([c.salida - c.servicio_inicio for c in clientes if c.salida is not None])
    idle_time_total = (2 * SIM_TIME) - total_servicio  # sumando idle time de 2 empleados
    promedio_idle = idle_time_total / 2.0  # idle time promedio por empleado
    
    resultado = {
        "total_clientes": len(clientes),
        "promedio_tiempo_total": statistics.mean(tiempos_totales) if tiempos_totales else 0,
        "prob_espera_mas_5": sum(1 for w in tiempos_espera if w > 5)/len(tiempos_espera) if tiempos_espera else 0,
        "promedio_idle_empleado": promedio_idle
    }
    return resultado

In [ ]:
# Simulacion y comparacion

print("Simulando Sistema 1 (dos colas independientes)...")
resultados_s1 = simular_sistema1()
print("Resultados Sistema 1:")
print(f"  Total de clientes atendidos (suma de ambas cajas): {resultados_s1['Sistema1']['total_clientes']}")
print(f"  Tiempo medio en el banco (min): {resultados_s1['Sistema1']['promedio_tiempo_total']:.2f}")
print(f"  Probabilidad de espera >5 min: {resultados_s1['Sistema1']['prob_espera_mas_5']:.2f}")
print(f"  Idle time medio por empleado (min): {resultados_s1['Sistema1']['promedio_idle_empleado']:.2f}")

print("\nSimulando Sistema 2 (cola única con 2 servidores)...")
resultados_s2 = simular_sistema2()
print("Resultados Sistema 2:")
print(f"  Total de clientes atendidos: {resultados_s2['total_clientes']}")
print(f"  Tiempo medio en el banco (min): {resultados_s2['promedio_tiempo_total']:.2f}")
print(f"  Probabilidad de espera >5 min: {resultados_s2['prob_espera_mas_5']:.2f}")
print(f"  Idle time medio por empleado (min): {resultados_s2['promedio_idle_empleado']:.2f}")


In [ ]:
# Visualización: Comparación de tiempos de espera y tiempos totales (histogramas)
def recolectar_tiempos_espera_s1():
    # Para cada caja:
    tiempos = []
    for caja in ["Caja_1", "Caja_2"]:
        data = resultados_s1["Detalle"][caja]
        # En este ejemplo, solo usaremos la media de cada caja (podríamos almacenar todos los tiempos si se deseara)
        tiempos.append(data["promedio_espera"])
    return tiempos

# Nota: En esta simulación se han registrado promedios globales; para histogramas más detallados
# se podría almacenar la lista de tiempos de espera de cada cliente en cada simulación.

labels = ['Sistema1 (colas separadas)', 'Sistema2 (cola única)']
mean_total = [resultados_s1['Sistema1']['promedio_tiempo_total'], resultados_s2['promedio_tiempo_total']]
idle_mean   = [resultados_s1['Sistema1']['promedio_idle_empleado'], resultados_s2['promedio_idle_empleado']]
prob_wait   = [resultados_s1['Sistema1']['prob_espera_mas_5'], resultados_s2['prob_espera_mas_5']]

x = np.arange(len(labels))
width = 0.25

fig, axs = plt.subplots(1, 3, figsize=(16,5))
axs[0].bar(x, mean_total, width, color='skyblue')
axs[0].set_ylabel("Tiempo medio total (min)")
axs[0].set_xticks(x)
axs[0].set_xticklabels(labels)
axs[0].set_title("Tiempo total en el banco")

axs[1].bar(x, prob_wait, width, color='salmon')
axs[1].set_ylabel("Probabilidad")
axs[1].set_xticks(x)
axs[1].set_xticklabels(labels, rotation=15)
axs[1].set_title("Probabilidad de esperar >5 min")

axs[2].bar(x, idle_mean, width, color='lightgreen')
axs[2].set_ylabel("Idle time (min)")
axs[2].set_xticks(x)
axs[2].set_xticklabels(labels)
axs[2].set_title("Idle time medio por empleado")

plt.tight_layout()
plt.show()